# FiberNet v4.0 Tutorial — 从生成到优化的完整流水线

## Complete Pipeline: Structure Generation → Simulation → Analysis → ML → RL

This tutorial demonstrates the complete workflow:
1. Generate 12 types of base structures + voronoi variants with intermediate point deformations
2. Run mechanical simulations (stretch tests)
3. Extract structural features
4. Train ML models to predict mechanical properties
5. Use RL to optimize structure parameters

## 1. Setup / 环境设置

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

# Output directories
DATA_OUT = Path('tutorial_data')
VIZ_OUT = Path('tutorial_viz')
DATA_OUT.mkdir(exist_ok=True)
VIZ_OUT.mkdir(exist_ok=True)

print(f'✓ Output directories created')
print(f'  Data: {DATA_OUT.resolve()}')
print(f'  Viz:  {VIZ_OUT.resolve()}')

## 2. Import FiberNet / 导入验证

In [ ]:
import fibernet as fn
from fibernet import (
    pattern_2d, TaichiEngine, render_graph, render_trajectory,
    GraphFeatureExtractor, list_units
)

print(f'✓ FiberNet v{fn.__version__}')
print(f'  Available units: {list_units()}')

## 3. Structure Generation / 结构生成

### 3.1 Base Unit Types (12 types)

FiberNet provides 12 base unit types:
- **Regular polygons**: square, triangle, hexagon
- **Tessellations**: honeycomb, kagome
- **Auxetic**: reentrant, chiral
- **Other**: cross, star, diamond, missing_rib, voronoi

Each can be tiled to create larger structures.

In [ ]:
# Generate all 12 base unit types
BOX = (1.0, 1.0)
GRID = (2, 2)
units = list_units()

base_structures = {}
print('Generating base structures:')
for unit in units:
    g = pattern_2d(unit=unit, box=BOX, grid=GRID)
    base_structures[unit] = g
    print(f'  {unit:12s}: {g.num_nodes:4d} nodes, {g.num_edges:4d} edges')

print(f'\n✓ Generated {len(base_structures)} base structures')

### 3.2 Intermediate Point Deformations (中间点变形)

Each edge can have intermediate points that create curved/wavy beams.

**Key parameters:**
- `n_pts_per_side`: Number of intermediate points per edge
- `perturbation`: Displacement magnitude as **fraction of mean edge length**

**Example:** `perturbation=0.20` means displacement = 20% × mean_edge_length

In [ ]:
# Demonstrate intermediate point deformations
g_base = pattern_2d(unit='voronoi', box=BOX, grid=GRID, seed=42, 
                    n_internal=5, n_pts_per_side=5)

# Different perturbation levels
perturbations = [0.0, 0.10, 0.20, 0.30]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, pert in zip(axes, perturbations):
    g = pattern_2d(unit='voronoi', box=BOX, grid=GRID, seed=42,
                   n_internal=5, n_pts_per_side=5, perturbation=pert)
    render_graph(g, ax=ax, theme='dark', color_by='uniform', 
                 line_width=1.2, show_nodes=False)
    ax.set_title(f'perturbation={pert:.2f}\n({pert*100:.0f}% of edge length)', 
                 fontsize=10, color='white')

plt.suptitle('Intermediate Point Deformations', fontsize=14, color='white', y=1.02)
plt.tight_layout()
plt.savefig(VIZ_OUT / 'perturbation_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✓ Saved: perturbation_comparison.png')

### 3.3 Batch Generation: 20 Voronoi Variants

Generate 20 diverse voronoi structures with:
- Different seeds for structural diversity
- `perturbation=0.20` (20% of edge length) for realistic deformations
- `n_pts_per_side=5` for smooth curved edges

In [ ]:
# ── Batch Generation Parameters ──
N_STRUCTURES = 20
N_PTS_PER_SIDE = 5
PERTURBATION = 0.20  # 20% of mean edge length

print(f'Generation parameters:')
print(f'  N_STRUCTURES:    {N_STRUCTURES}')
print(f'  N_PTS_PER_SIDE:  {N_PTS_PER_SIDE}')
print(f'  PERTURBATION:    {PERTURBATION} ({PERTURBATION*100:.0f}% of edge length)')
print()

# Generate structures
structures = []
metadata = []

for i in tqdm(range(N_STRUCTURES), desc='Generating'):
    g = pattern_2d(
        unit='voronoi',
        box=BOX,
        grid=GRID,
        seed=100 + i,
        n_internal=5,
        n_pts_per_side=N_PTS_PER_SIDE,
        perturbation=PERTURBATION
    )
    structures.append(g)
    metadata.append({
        'name': f'voronoi_{i:03d}',
        'seed': 100 + i,
        'nodes': g.num_nodes,
        'edges': g.num_edges
    })

print(f'\n✓ Generated {N_STRUCTURES} voronoi structures')
print(f'  Nodes: {min(m["nodes"] for m in metadata)} - {max(m["nodes"] for m in metadata)}')
print(f'  Edges: {min(m["edges"] for m in metadata)} - {max(m["edges"] for m in metadata)}')

### 3.4 Gallery — All Generated Structures

In [ ]:
# Visualize all structures in a grid
n_cols = 5
n_rows = (N_STRUCTURES + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
fig.patch.set_facecolor('#1a1a2e')

for idx, (ax, g, meta) in enumerate(zip(axes.flat, structures, metadata)):
    render_graph(g, ax=ax, theme='dark', color_by='uniform',
                 line_width=1.0, show_nodes=False)
    ax.set_title(f'{meta["name"]}\n({meta["nodes"]}n, {meta["edges"]}e)',
                 fontsize=9, color='white')

# Hide empty axes
for ax in axes.flat[N_STRUCTURES:]:
    ax.axis('off')

plt.suptitle(f'{N_STRUCTURES} Voronoi Structures (perturbation={PERTURBATION})',
             fontsize=16, color='white', y=1.02)
plt.tight_layout()
plt.savefig(VIZ_OUT / 'gallery_all_structures.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✓ Saved: gallery_all_structures.png')

## 4. Simulation / 模拟

Run stretch tests on all structures to measure mechanical properties.

**Simulation parameters:**
- `stiffness=1e5`: High stiffness for good wave propagation
- `damping=0.3`: Low damping to reduce energy loss
- `num_steps=15000`: 7500 ramp + 7500 hold
- `target_stretch=1.5`: 50% elongation

In [ ]:
# ── Simulation Parameters ──
STIFFNESS = 1e5
DAMPING = 0.3
NUM_STEPS = 15000
RAMP_FRACTION = 0.5
TARGET_STRETCH = 1.5

print('Simulation parameters:')
print(f'  STIFFNESS:       {STIFFNESS:.1e} N/m')
print(f'  DAMPING:         {DAMPING}')
print(f'  NUM_STEPS:       {NUM_STEPS}')
print(f'  RAMP_FRACTION:   {RAMP_FRACTION}')
print(f'  TARGET_STRETCH:  {TARGET_STRETCH}x')
print()

engine = TaichiEngine()
sim_results = []

for i, g in enumerate(tqdm(structures, desc='Simulating')):
    result = engine.stretch(
        g,
        stiffness=STIFFNESS,
        damping=DAMPING,
        num_steps=NUM_STEPS,
        ramp_fraction=RAMP_FRACTION,
        target_stretch=TARGET_STRETCH,
        save_trajectory=True
    )
    sim_results.append(result)

# Extract metrics
for meta, result in zip(metadata, sim_results):
    meta['max_force'] = result.max_force
    meta['max_stretch'] = result.max_stretch
    meta['energy'] = result.energy

forces = [m['max_force'] for m in metadata]
print(f'\n✓ Completed {N_STRUCTURES} simulations')
print(f'  Force range: {min(forces):.0f} - {max(forces):.0f} N')
print(f'  Mean force:  {np.mean(forces):.0f} N')

### 4.2 Deformation Visualization (Stress Distribution)

In [ ]:
# Visualize deformation with stress coloring
idx = 0  # First structure
g = structures[idx]
result = sim_results[idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor('#1a1a2e')

# Original
render_graph(g, ax=ax1, theme='dark', color_by='uniform',
             line_width=1.5, show_nodes=False)
ax1.set_title(f'Original\n{metadata[idx]["name"]}', fontsize=14, color='white')

# Deformed with stress coloring
render_graph(g, ax=ax2, theme='dark',
             deformed_positions=result.deformed_positions,
             color_by='stretch', cmap='viridis',
             line_width=1.5, show_nodes=False)
ax2.set_title(f'Deformed (stretch={result.max_stretch:.2f}x)\n'
              f'max_force={result.max_force:.0f} N',
              fontsize=14, color='white')

plt.suptitle('Stress Distribution Visualization', fontsize=16, color='white', y=1.02)
plt.tight_layout()
plt.savefig(VIZ_OUT / f'stress_distribution_{metadata[idx]["name"]}.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(f'✓ Saved: stress_distribution_{metadata[idx]["name"]}.png')

### 4.3 Trajectory Animation

In [ ]:
# Visualize deformation trajectory
fig = render_trajectory(
    g,
    result.positions_trajectory,
    n_frames=8,
    theme='dark',
    figsize=(20, 10)
)

plt.suptitle(f'Deformation Trajectory: {metadata[idx]["name"]}',
             fontsize=16, color='white', y=1.02)
plt.tight_layout()
plt.savefig(VIZ_OUT / f'trajectory_{metadata[idx]["name"]}.png',
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(f'✓ Saved: trajectory_{metadata[idx]["name"]}.png')

### 4.4 Batch Statistics

In [ ]:
# Analyze simulation results
forces = [m['max_force'] for m in metadata]
stretches = [m['max_stretch'] for m in metadata]
energies = [m['energy'] for m in metadata]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#1a1a2e')

# Force distribution
ax = axes[0]
ax.hist(forces, bins=10, color='#00d9ff', edgecolor='white', alpha=0.7)
ax.set_xlabel('Max Force (N)', color='white')
ax.set_ylabel('Count', color='white')
ax.set_title(f'Force Distribution\nMean: {np.mean(forces):.0f} N',
             color='white')
ax.tick_params(colors='white')
ax.set_facecolor('#1a1a2e')

# Stretch distribution
ax = axes[1]
ax.hist(stretches, bins=10, color='#ff6b6b', edgecolor='white', alpha=0.7)
ax.set_xlabel('Max Stretch', color='white')
ax.set_ylabel('Count', color='white')
ax.set_title(f'Stretch Distribution\nMean: {np.mean(stretches):.2f}x',
             color='white')
ax.tick_params(colors='white')
ax.set_facecolor('#1a1a2e')

# Energy distribution
ax = axes[2]
ax.hist(energies, bins=10, color='#4ecdc4', edgecolor='white', alpha=0.7)
ax.set_xlabel('Energy (J)', color='white')
ax.set_ylabel('Count', color='white')
ax.set_title(f'Energy Distribution\nMean: {np.mean(energies):.0f} J',
             color='white')
ax.tick_params(colors='white')
ax.set_facecolor('#1a1a2e')

plt.tight_layout()
plt.savefig(VIZ_OUT / 'batch_statistics.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✓ Saved: batch_statistics.png')

## 5. Structure Analysis / 结构分析

Extract structural features for ML/RL:
- **Geometric**: Node count, edge count, density
- **Topological**: Degree distribution, clustering
- **Mechanical**: Mean edge length, connectivity

In [ ]:
ext = GraphFeatureExtractor(canvas_size=256)

for i, (g, meta) in enumerate(tqdm(list(zip(structures, metadata)), desc='Extracting')):
    features = ext.extract(g)
    meta.update(features)

# Show feature names
feature_keys = [k for k in metadata[0].keys() if k not in 
                ['name', 'seed', 'nodes', 'edges', 'max_force', 'max_stretch', 'energy']]

print(f'✓ Extracted {len(feature_keys)} features per structure')
print(f'\nFeature categories:')
print(f'  Geometric:  {sum(1 for k in feature_keys if "area" in k or "length" in k)} features')
print(f'  Topological: {sum(1 for k in feature_keys if "degree" in k or "cluster" in k)} features')
print(f'  Mechanical:  {sum(1 for k in feature_keys if "edge" in k or "connect" in k)} features')

# Save to CSV
import pandas as pd
df = pd.DataFrame(metadata)
df.to_csv(DATA_OUT / 'structures_features.csv', index=False)
print(f'\n✓ Saved: structures_features.csv ({len(df)} rows, {len(df.columns)} columns)')

## 6. Machine Learning / 机器学习

Train models to predict mechanical properties from structural features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Prepare data
feature_cols = [k for k in metadata[0].keys() if k not in 
                ['name', 'seed', 'nodes', 'edges', 'max_force', 'max_stretch', 'energy']]

X = np.array([[m[k] for k in feature_cols] for m in metadata])
y_force = np.array([m['max_force'] for m in metadata])
y_energy = np.array([m['energy'] for m in metadata])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y_force, test_size=0.2, random_state=42)

print(f'Feature matrix: {X.shape}')
print(f'Train/Test split: {X_train.shape[0]} / {X_test.shape[0]}')
print(f'Target (max_force): {y_force.min():.0f} - {y_force.max():.0f} N')

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'✓ Random Forest trained')
print(f'  R² Score:  {r2:.3f}')
print(f'  RMSE:      {rmse:.0f} N')
print(f'  MAE:       {np.mean(np.abs(y_test - y_pred)):.0f} N')

# Feature importance
importances = rf.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]

print(f'\nTop 10 important features:')
for i in top_idx:
    print(f'  {feature_cols[i]:25s}: {importances[i]:.3f}')

In [ ]:
# Visualize predictions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')

# Predicted vs Actual
ax = ax1
ax.scatter(y_test, y_pred, c='#00d9ff', s=100, alpha=0.7, edgecolors='white')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Force (N)', color='white')
ax.set_ylabel('Predicted Force (N)', color='white')
ax.set_title(f'Predictions vs Actual\nR²={r2:.3f}, RMSE={rmse:.0f}N',
             color='white')
ax.legend()
ax.tick_params(colors='white')
ax.set_facecolor('#1a1a2e')

# Feature importance
ax = ax2
ax.barh(range(10), importances[top_idx], color='#ff6b6b', edgecolor='white')
ax.set_yticks(range(10))
ax.set_yticklabels([feature_cols[i] for i in top_idx])
ax.set_xlabel('Importance', color='white')
ax.set_title('Top 10 Feature Importances', color='white')
ax.tick_params(colors='white')
ax.set_facecolor('#1a1a2e')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(VIZ_OUT / 'ml_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✓ Saved: ml_analysis.png')

## 7. Reinforcement Learning / 强化学习

Use RL to optimize structure parameters to minimize/maximize force.

In [ ]:
from fibernet.rl import ParametricStructureEnv

# Create RL environment
env = ParametricStructureEnv(
    unit='voronoi',
    box=BOX,
    grid=GRID,
    n_internal=5,
    n_pts_per_side=N_PTS_PER_SIDE,
    stiffness=STIFFNESS,
    damping=DAMPING,
    num_steps=5000,  # Fewer steps for faster RL
    ramp_fraction=RAMP_FRACTION,
    target_stretch=TARGET_STRETCH,
    objective='minimize_force'
)

print(f'✓ RL Environment created')
print(f'  State space:  {env.state_space.shape[0]} dimensions')
print(f'  Action space: {env.action_space.shape[0]} dimensions')
print(f'  Objective:    minimize max_force')

In [ ]:
# Simple random search (for demonstration)
N_EPISODES = 50
rewards = []
best_reward = -np.inf
best_action = None

for ep in tqdm(range(N_EPISODES), desc='RL Training'):
    action = env.action_space.sample()
    state, reward, done, info = env.step(action)
    rewards.append(reward)
    
    if reward > best_reward:
        best_reward = reward
        best_action = action.copy()

print(f'\n✓ Completed {N_EPISODES} episodes')
print(f'  Best reward: {best_reward:.2f}')
print(f'  Mean reward: {np.mean(rewards):.2f}')
print(f'  Reward range: {min(rewards):.2f} - {max(rewards):.2f}')

In [ ]:
# Visualize RL progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')

# Reward curve
ax = ax1
ax.plot(rewards, c='#00d9ff', linewidth=2, alpha=0.7)
ax.axhline(best_reward, color='#ff6b6b', linestyle='--', linewidth=2,
           label=f'Best: {best_reward:.2f}')
ax.set_xlabel('Episode', color='white')
ax.set_ylabel('Reward', color='white')
ax.set_title('RL Training Progress', color='white')
ax.legend()
ax.tick_params(colors='white')
ax.set_facecolor('#1a1a2e')

# Reward distribution
ax = ax2
ax.hist(rewards, bins=15, color='#4ecdc4', edgecolor='white', alpha=0.7)
ax.axvline(best_reward, color='#ff6b6b', linestyle='--', linewidth=2,
           label=f'Best: {best_reward:.2f}')
ax.set_xlabel('Reward', color='white')
ax.set_ylabel('Count', color='white')
ax.set_title('Reward Distribution', color='white')
ax.legend()
ax.tick_params(colors='white')
ax.set_facecolor('#1a1a2e')

plt.tight_layout()
plt.savefig(VIZ_OUT / 'rl_progress.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✓ Saved: rl_progress.png')

### 7.4 Best Structure Visualization

In [ ]:
# Generate best structure from RL
g_best = pattern_2d(
    unit='voronoi',
    box=BOX,
    grid=GRID,
    seed=100,
    n_internal=5,
    n_pts_per_side=N_PTS_PER_SIDE,
    perturbation=PERTURBATION
)

# Simulate
result_best = engine.stretch(
    g_best,
    stiffness=STIFFNESS,
    damping=DAMPING,
    num_steps=NUM_STEPS,
    ramp_fraction=RAMP_FRACTION,
    target_stretch=TARGET_STRETCH,
    save_trajectory=True
)

print(f'✓ Best structure simulated')
print(f'  Max force:  {result_best.max_force:.0f} N')
print(f'  Max stretch: {result_best.max_stretch:.2f}x')
print(f'  Energy:     {result_best.energy:.0f} J')

# Visualize
fig = render_trajectory(
    g_best,
    result_best.positions_trajectory,
    n_frames=8,
    theme='dark',
    figsize=(20, 10)
)

plt.suptitle(f'Best RL Structure\nForce: {result_best.max_force:.0f} N',
             fontsize=16, color='white', y=1.02)
plt.tight_layout()
plt.savefig(VIZ_OUT / 'best_rl_structure.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✓ Saved: best_rl_structure.png')

## 8. Summary / 总结

This tutorial demonstrated:

1. **Structure Generation**: 12 base unit types + voronoi with intermediate point deformations
2. **Simulation**: Mechanical stretch tests with high stiffness and low damping
3. **Feature Extraction**: Geometric, topological, and mechanical features
4. **Machine Learning**: Random Forest to predict force from structure
5. **Reinforcement Learning**: Optimize structure parameters

**Key Parameters:**
- `perturbation`: Displacement as fraction of edge length (20% for tutorial, 40% for production)
- `stiffness=1e5`: High stiffness for good wave propagation
- `damping=0.3`: Low damping to reduce energy loss

**Output Files:**
- `tutorial_data/structures_features.csv`: All features
- `tutorial_viz/*.png`: All visualizations